# Sentiment Analysis using NLP Pipeline & Machine Learning

##  Objective
The objective of this project is to build an end-to-end Sentiment Analysis system using Natural Language Processing (NLP) techniques and Machine Learning models. The project includes preprocessing raw text data, converting it into numerical features, training multiple models, and evaluating their performance.

# STEP 1: Setup and Load Dataset

## 1. Import Libraries

In [6]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## 2. Load Dataset from Colab

In [7]:
df = pd.read_csv('/content/balanced_sentiment_dataset.csv', engine='python', on_bad_lines='skip')

## 3. Basic Inspection

In [8]:
# First 5 rows
df.head()

,sentiment,text
0,1,"And here is the rap song ""African Warrior Quee..."
1,1,"We asked chatGPT ,\n\nHow to become a successf..."
2,0,When I finally manage to convince myself to pe...
3,0,Although a healthier alternative to your typic...
4,1,"In my latest test, I asked #ChatGPT to write m..."


In [9]:
# Dataset shape
df.shape

(80000, 2)

In [10]:
# Column names
df.columns

Index(['sentiment', 'text'], dtype='object')

In [11]:
# Check missing values
df.isnull().sum()

,0
sentiment,0
text,0


## 4. Convert Labels

In [12]:
df['sentiment'] = df['sentiment'].map({0: 'negative', 1: 'positive'})

## 5. Class Distribution

In [13]:
df['sentiment'].value_counts()

,count
sentiment,
positive,40000
negative,40000


## 6. Sample Texts

In [14]:
print("Sample Reviews:\n")
for i in range(5):
    print(df['text'][i])
    print("Sentiment:", df['sentiment'][i])
    print("------")

Sample Reviews:

And here is the rap song "African Warrior Queens", for which ChatGPT wrote the lyrics 🤎 Yes, amateur but beautiful :)\n\n1/1 Ξ 0.1 on KO ⚔️ link below 🔊 sound on https://t.co/cyHY3m2qHy
Sentiment: positive
------
We asked chatGPT ,\n\nHow to become a successful trader?\n\nHere is what it thinks,\n\n#ChatGPT https://t.co/UKfbOqyurW
Sentiment: positive
------
When I finally manage to convince myself to peel out from under the sheets each morning the second thing I'm interested in is a good strong cup of coffee. I categorize myself as a coffee drinker as opposed to a connoisseur since I'm amenable to buying whatever happens to be on sale at the market and spending my money on boutique blends happens about as often as the pope personally dispenses contraceptives.<br /><br />Now, my palate is sufficiently sophisticated to distinguish between premium and plebian but the miserly gene invariably kicks into overdrive and inhibits forking over the cash for the former and definit

# STEP 2: NLP Preprocessing

## 1. Initialize NLP Tools

In [15]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

## 2. Create Preprocessing Function

In [16]:
def preprocess_text(text):

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # 3. Remove special characters & numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 4. Tokenization
    words = text.split()

    # 5. Remove stopwords
    words = [word for word in words if word not in stop_words]

    # 6. Lemmatization (better than stemming)
    words = [lemmatizer.lemmatize(word) for word in words]

    # Join back
    return " ".join(words)

## 3. Apply Preprocessing to Dataset

In [17]:
df['clean_text'] = df['text'].apply(preprocess_text)

## 4. Compare Before vs After

In [18]:
for i in range(3):
    print("Original:", df['text'][i])
    print("Cleaned :", df['clean_text'][i])
    print("------")

Original: And here is the rap song "African Warrior Queens", for which ChatGPT wrote the lyrics 🤎 Yes, amateur but beautiful :)\n\n1/1 Ξ 0.1 on KO ⚔️ link below 🔊 sound on https://t.co/cyHY3m2qHy
Cleaned : rap song african warrior queen chatgpt wrote lyric yes amateur beautiful nn ko link sound
------
Original: We asked chatGPT ,\n\nHow to become a successful trader?\n\nHere is what it thinks,\n\n#ChatGPT https://t.co/UKfbOqyurW
Cleaned : asked chatgpt nnhow become successful tradernnhere thinksnnchatgpt
------
Original: When I finally manage to convince myself to peel out from under the sheets each morning the second thing I'm interested in is a good strong cup of coffee. I categorize myself as a coffee drinker as opposed to a connoisseur since I'm amenable to buying whatever happens to be on sale at the market and spending my money on boutique blends happens about as often as the pope personally dispenses contraceptives.<br /><br />Now, my palate is sufficiently sophisticated to dist

## 5. Check Result

In [19]:
df.head()

,sentiment,text,clean_text
0,positive,"And here is the rap song ""African Warrior Quee...",rap song african warrior queen chatgpt wrote l...
1,positive,"We asked chatGPT ,\n\nHow to become a successf...",asked chatgpt nnhow become successful tradernn...
2,negative,When I finally manage to convince myself to pe...,finally manage convince peel sheet morning sec...
3,negative,Although a healthier alternative to your typic...,although healthier alternative typical vanilla...
4,positive,"In my latest test, I asked #ChatGPT to write m...",latest test asked chatgpt write welcome letter...


# STEP 3: Feature Engineering

## 1. Define Input (X) and Output (y)

In [20]:
X = df['clean_text']
y = df['sentiment']

## 2. Train-Test Split

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## **Bag of Words (BoW)**

###  Apply BoW

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

### Check Shape

In [23]:
print("BoW Train Shape:", X_train_bow.shape)
print("BoW Test Shape :", X_test_bow.shape)

BoW Train Shape: (64000, 5000)
BoW Test Shape : (16000, 5000)


## **TF-IDF**

### Apply TF-IDF

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

### Check Shape

In [25]:
print("TF-IDF Train Shape:", X_train_tfidf.shape)
print("TF-IDF Test Shape :", X_test_tfidf.shape)

TF-IDF Train Shape: (64000, 5000)
TF-IDF Test Shape : (16000, 5000)


# STEP 4: Model Building

## **Models on BoW**

### 1. Logistic Regression (BoW)

In [26]:
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression()
lr_model.fit(X_train_bow, y_train)

y_pred_lr_bow = lr_model.predict(X_test_bow)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 2. Naive Bayes (BoW)

In [27]:
from sklearn.naive_bayes import MultinomialNB
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

y_pred_nb_bow = nb_model.predict(X_test_bow)

### 3. Decision Tree (BoW)

In [28]:
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train_bow, y_train)

y_pred_dt_bow = dt_model.predict(X_test_bow)

## **Models on TF-IDF**

### 1. Logistic Regression (TF-IDF)

In [29]:
lr_model_tfidf = LogisticRegression()
lr_model_tfidf.fit(X_train_tfidf, y_train)

y_pred_lr_tfidf = lr_model_tfidf.predict(X_test_tfidf)

### 2. Naive Bayes (TF-IDF)

In [30]:
nb_model_tfidf = MultinomialNB()
nb_model_tfidf.fit(X_train_tfidf, y_train)

y_pred_nb_tfidf = nb_model_tfidf.predict(X_test_tfidf)

### 3. Decision Tree (TF-IDF)

In [31]:
dt_model_tfidf = DecisionTreeClassifier()
dt_model_tfidf.fit(X_train_tfidf, y_train)

y_pred_dt_tfidf = dt_model_tfidf.predict(X_test_tfidf)

# STEP 5: Model Evaluation

## 1. Create Evaluation Function

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label='positive')
    rec = recall_score(y_true, y_pred, pos_label='positive')
    f1 = f1_score(y_true, y_pred, pos_label='positive')

    return acc, prec, rec, f1

## 2. Evaluate BoW Models

In [33]:
results = []

# Logistic Regression (BoW)
results.append(("LR BoW", *evaluate_model(y_test, y_pred_lr_bow)))

# Naive Bayes (BoW)
results.append(("NB BoW", *evaluate_model(y_test, y_pred_nb_bow)))

# Decision Tree (BoW)
results.append(("DT BoW", *evaluate_model(y_test, y_pred_dt_bow)))

## 3. Evaluate TF-IDF Models

In [34]:
# Logistic Regression (TF-IDF)
results.append(("LR TF-IDF", *evaluate_model(y_test, y_pred_lr_tfidf)))

# Naive Bayes (TF-IDF)
results.append(("NB TF-IDF", *evaluate_model(y_test, y_pred_nb_tfidf)))

# Decision Tree (TF-IDF)
results.append(("DT TF-IDF", *evaluate_model(y_test, y_pred_dt_tfidf)))

## 4. Create Comparison Table

In [35]:
results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score"
])

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,LR BoW,0.799438,0.810399,0.777624,0.793673
1,NB BoW,0.745250,0.722947,0.788711,0.754399
2,DT BoW,0.740688,0.737641,0.740708,0.739171
3,LR TF-IDF,0.816750,0.821039,0.806350,0.813628
4,NB TF-IDF,0.785188,0.774792,0.799294,0.786853
5,DT TF-IDF,0.730688,0.731673,0.721809,0.726708


## 5. Find Best Model

In [36]:
best_model = results_df.sort_values(by="F1 Score", ascending=False)
best_model

,Model,Accuracy,Precision,Recall,F1 Score
3,LR TF-IDF,0.816750,0.821039,0.806350,0.813628
0,LR BoW,0.799438,0.810399,0.777624,0.793673
4,NB TF-IDF,0.785188,0.774792,0.799294,0.786853
1,NB BoW,0.745250,0.722947,0.788711,0.754399
2,DT BoW,0.740688,0.737641,0.740708,0.739171
5,DT TF-IDF,0.730688,0.731673,0.721809,0.726708


# STEP 6: Comparison & Insights

## **Model Comparison & Insights**

### 1. Best Performing Model:
From the evaluation table, the model with the highest F1 Score is Logistic Regression with TF-IDF.    
This model provides the best balance between precision and recall.

### 2. Effect of Preprocessing:
Text preprocessing significantly improved model performance by:
*   Removing noise (punctuation, URLs, special characters)
*   Eliminating stopwords
*   Standardizing words using lemmatization.

This helped models learn meaningful patterns instead of irrelevant data.

### 3. BoW vs TF-IDF:

*   **Bag of Words (BoW)**: Counts word frequency but does not consider importance
*  **TF-IDF**: Assigns importance to words based on their uniqueness



### 4. Model Comparison:

*   **Logistic Regression**: Performed best due to its effectiveness in text classification
*   **Naive Bayes**: Fast and simple but slightly lower performance
*   **Decision Tree**: Tends to overfit and performed comparatively lower



### 5. Trade-offs:

*   Simpler models (Naive Bayes) are faster but less accurate
*   Complex models (Decision Tree) may overfit
*   Logistic Regression provides a good balance of performance and stability



### 6. Conclusion:
The combination of TF-IDF + Logistic Regression gives the best performance for sentiment analysis in this project.

#  Final Conclusion

In this project, we successfully built an end-to-end Sentiment Analysis system using NLP techniques and Machine Learning models.                           

Key takeaways:                      
- Text preprocessing plays a crucial role in improving model performance.            
- TF-IDF vectorization outperformed Bag of Words by capturing word importance.
- Logistic Regression achieved the best performance among all models.
- Model evaluation using Accuracy, Precision, Recall, and F1 Score provided a clear comparison.       

Overall, this project demonstrates how raw text data can be transformed into meaningful insights using NLP and machine learning.                     